In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "0" 
os.environ["GRB_LICENSE_FILE"] = "/usr0/home/naveenr/gurobi.lic" 
os.environ['MKL_THREADING_LAYER'] = "GNU"
import torch 

/usr0/home/naveenr/miniconda3/envs/food/lib/python3.10/site-packages/torch/cuda/__init__.py:54: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
from concept_abstraction.training import *
from concept_abstraction.selection import *
from concept_abstraction.concept_bank import *
from concept_abstraction.env_utils import *
from concept_abstraction.environments import *
from concept_abstraction.utils import *
from concept_abstraction.environments import ConceptEnv
import sys 
import argparse
import secrets
import numpy as np 
import random 
import os
from stable_baselines3 import PPO
import pickle
import resource
import time 

In [4]:
is_jupyter = 'ipykernel' in sys.modules
is_main = __name__ == "__main__"

In [17]:
if is_main:
    seed = 43
    environment_string = "cart_pole"
    gold_timesteps = 4_000_000
    training_timesteps = 250_000 
    num_concepts_selected = 3
    out_folder = "basic"
    method = "lp" 


In [64]:
if is_main:
    concept_list, processed_concepts = get_concepts(environment_string,"human_selected_binary",seed)
    num_concepts_selected = min(num_concepts_selected,len(concept_list))
    ground_truth_env, ground_truth_gym_env = get_environment(environment_string, None, seed)   
    model_name = "../../results/models/env={}_training={}_seed={}.zip".format(environment_string,gold_timesteps,seed)
    if os.path.exists(model_name):
        groundtruth_model = PPO.load(model_name)
    model_name = "../../results/q_estimates/env={}_training={}_seed={}_selection={}_source={}.pkl".format(environment_string,gold_timesteps,seed,"q_value","human_selected_binary")
    if os.path.exists(model_name):
        q_estimates = pickle.load(open(model_name,"rb"))

In [20]:
subset_concept, idx = policy_coverage_selection(ground_truth_gym_env,concept_list,num_concepts_selected,groundtruth_model)
idx 

Coverage 0.8657888427109974


[6, 9, 7]

In [14]:
acc_list = [np.random.random() for i in range(len(concept_list))]
subset_concept, idx = policy_coverage_selection_exp_lp(ground_truth_gym_env,concept_list,acc_list,num_concepts_selected,groundtruth_model)
json.dumps(idx)

'[0,6,12,19,30,35,37,38,39,42,43]'

In [23]:
full_two_stage_env, full_two_stage_gym_env = get_environment(environment_string,concept_list,seed,processed_concepts=processed_concepts,concept_idx=idx)
all_concept_model = train_ppo_model(full_two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=150_000,custom_name="{}_perfect_greedy_{}".format(environment_string,seed)) 

wandb: Currently logged in as: naveenr (naveenr-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


approx_kl,█▆▄▄▃▃▃▃▄▇▁▄▁▄▂▂▂▂▂▂▂▁▁▂▂▂▂▄▂▂▂▂▂▂▂▂▁▁▂▁
clip_fraction,█▄▄▄▃▁▁▁▁▁▂▂▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▃▁▁▁▂▂▂
ema_norm_reward,▁▁▁▁▁▁▁▁▁▂▂▂▂▃▃▄▄▅▅▅▆▅▅▅▅▇▇▇▇▇▇▇███▇▇▇▆▆
entropy_loss,▁▁▁▂▂▃▄▄▄▄▅▅▅▆▆▆▆▇▇▇██▇▇▇▇▇▇██▇▇██▇▇▆▆▆▇
episode_length_mean,▁▁▁▁▁▁▁▁▂▁▂▂▃▁▁▄▁▄▃▃▂▄▂▄▆▇█▆▂█▄▅█▆▇▅▇█▄▆
episode_reward_max,▁▁▂▁▁▃▂▃▃▂▅▃▂▄▅▄▂▄▄█▃▆▆▆█▇▅▇████▇█▇█▆██▂
episode_reward_mean,▁▁▁▁▁▂▂▁▂▂▂▂▂▃▄▆▄▆▄▇▅▇▅▄▅█▅██▇██▄█▅▇█▅██
episode_reward_min,▁▁▁▁▁▁▁▁▂▁▂▃▁▃▄▆▂▄▇▄▃▆▃█▆▅▃█▅▇███▄▅█▆▅▅▆
episodes_completed,▁▁▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███████
explained_variance,████▇█████████████████████████▁█▇██▁████
+1,...


In [66]:
ground_truth_env.reset()

array([[[[255, 255, 255, ..., 255, 255, 255],
         [255, 255, 255, ..., 255, 255, 255],
         [255, 255, 255, ..., 255, 255, 255],
         ...,
         [255, 255, 255, ..., 255, 255, 255],
         [255, 255, 255, ..., 255, 255, 255],
         [255, 255, 255, ..., 255, 255, 255]],

        [[255, 255, 255, ..., 255, 255, 255],
         [255, 255, 255, ..., 255, 255, 255],
         [255, 255, 255, ..., 255, 255, 255],
         ...,
         [255, 255, 255, ..., 255, 255, 255],
         [255, 255, 255, ..., 255, 255, 255],
         [255, 255, 255, ..., 255, 255, 255]],

        [[255, 255, 255, ..., 255, 255, 255],
         [255, 255, 255, ..., 255, 255, 255],
         [255, 255, 255, ..., 255, 255, 255],
         ...,
         [255, 255, 255, ..., 255, 255, 255],
         [255, 255, 255, ..., 255, 255, 255],
         [255, 255, 255, ..., 255, 255, 255]],

        [[255, 255, 255, ..., 255, 255, 255],
         [255, 255, 255, ..., 255, 255, 255],
         [255, 255, 255, ..., 25

In [70]:
def make_env():
    env = Monitor(gym.make("CartPole-v1"))
    env = ConceptWrapper(env,spaces.Box(
            low=0, high=255,
            shape=(4,),  # Height x Width, no color channel
            dtype=np.uint8
        ),get_raw_state_cartpole)
    return env 
e = DummyVecEnv([make_env for i in range(8)])
e.reset()
deepcopy(e)

In [73]:
wandb.finish()

approx_kl,▁▁▁▁▁▁▁▁▁▁▁
clip_fraction,▁▁▁▁▁▁▁▁▁▁▁
ema_norm_reward,▁▁▃▃▄▄▄▅▆▅▅▆▆▅▄▄▄▅▄▄▄▄▄▅▅▅▅▇███▇▇▇
entropy_loss,▁▁▁▁▁▁▁▁▁▁▁
episode_length_mean,▁▂▃▂▃▂▃▃▄▁▃▅▂▂▂▂▂▃▂▂▃▁▃▄▃▄▄█▇▅▅▄▅▄
episode_reward_max,▁▂▂▂▄▁▂▃▅▁▃█▃▂▂▂▂▃▃▃▃▁▃▃▄▅▃▇▆▄▅▅▆▃
episode_reward_mean,▁▂▃▂▃▂▃▃▄▁▃▅▂▂▂▂▂▃▂▂▃▁▃▄▃▄▄█▇▅▅▄▅▄
episode_reward_min,▁▂▂▁▁▂▄▁▁▂▂▂▂▂▂▂▂▂▁▁▂▁▁▅▂▁▃▇█▇▅▂▂▄
episodes_completed,▁▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
explained_variance,▁▁▁▁▁▁▁▁▁▁▁
+1,...


In [96]:
pickle.dump(1,open("temp.pkl","wb"))

approx_kl,██▅▅▄▃▂▄▄▆▄▄▁▂▃▄▂▆▂▃▁▁▃▃▂▄▂▂▂▄▁▁▁▁▃▂▂▆▃▃
clip_fraction,▃▃▄▄▆▄▅█▂▃▃▁▆▇▂▂▃▂▂▂▃▃▇▅▆▃▅▅▄▃▄▂▂▁▄▂▆▆▆▅
ema_norm_reward,▁▁▁▁▁▁▁▁▁▂▂▃▃▄▄▅▅▆▇▆▅▅▆▅▆▇▇▇▇▆██▇▅█▇██▇█
entropy_loss,▁▂▂▂▆▅▅▄▅▄▆▆▇▇▇▆▇▇█▇▇▇▇▇▇▇▇▇▇█▇▇▇▇██████
episode_length_mean,▁▁▁▁▁▁▁▂▂▅█▄▇▅▅█▇▄▄▅▄▅▆▆▆▇▄███▅▇███▅▆██▇
episode_reward_max,▁▁▁▁▁▁▁▂▂▂▄▄▇▄▄▇█▄▆█▆█▄██▅▄▃▄██▅▆█▇█▆███
episode_reward_mean,▁▁▁▂▁▂▂▂▆▄▇▄▄▆▄▃▃▃▃▃▄▆▄▇▄▆▇███▆▃██▆▇███▇
episode_reward_min,▁▁▁▂▁▆▄▄█▆▆▅▂▃██▄█▇▇▄█▅██▅▇█████▆▆██████
episodes_completed,▁▁▁▁▁▂▂▂▂▃▃▄▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██
explained_variance,▆▇▆▆▇▆▇▆▆▆▇▇▇▇▇▆▇▇▇▇▇▇▇▇▇▇▆▆▇▇▇▇▇██▁▅▆▆▇
+1,...


Episode 1
Episode 2
Episode 3
Episode 4
Episode 5
Episode 6
Episode 7
Episode 8
Episode 9
Episode 10
Episode 11
Episode 12
Episode 13
Episode 14
Episode 15
Episode 16
Episode 17
Episode 18
Episode 19
Episode 20
Episode 21
Episode 22
Episode 23
Episode 24
Episode 25
Episode 26
Episode 27
Episode 28
Episode 29
Episode 30
Episode 31
Episode 32
Episode 33
Episode 34
Episode 35
Episode 36
Episode 37
Episode 38
Episode 39
Episode 40
Episode 41
Episode 42
Episode 43
Episode 44
Episode 45
Episode 46
Episode 47
Episode 48
Episode 49
Episode 50
[{}, {}, {}, {}, {}, {}, {}, {}]
On rollout 0
On rollout 1
On rollout 2
On rollout 3
On rollout 4
On rollout 5
On rollout 6
On rollout 7
On rollout 8
On rollout 9
On rollout 10
On rollout 11
On rollout 12
On rollout 13
On rollout 14
On rollout 15
On rollout 16
On rollout 17
On rollout 18
On rollout 19
On rollout 20
On rollout 21
On rollout 22
On rollout 23
On rollout 24
On rollout 25
On rollout 26
On rollout 27
On rollout 28
On rollout 29
On rollout 30
On

In [306]:
class TDTrainer:
    def __init__(self, value_net, env, policy, gamma=0.995, lr=1e-4, lambda_=0.95, device='cuda'):
        self.value_net = value_net.to(device)
        self.env = env
        self.policy = policy
        self.gamma = gamma
        self.lambda_ = lambda_
        self.device = device
        self.optimizer = optim.Adam(self.value_net.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()

    def collect_rollouts(self, num_steps):
        """
        Interacts with the VecEnv to collect 'num_steps' of data per environment.
        Returns flattened tensors ready for training.
        """
        batch_obs = []
        batch_rewards = []
        batch_dones = []

        if not hasattr(self, 'last_obs'):
            self.last_obs = self.env.reset()

        for _ in range(num_steps):
            action, _ = self.policy.predict(self.last_obs, deterministic=False)
            next_obs, rewards, dones, infos = self.env.step(action)
            
            batch_obs.append(self.last_obs.copy()) 
            batch_rewards.append(rewards)
            batch_dones.append(dones)
            
            self.last_obs = next_obs

        obs_arr = np.array(batch_obs)  
        rew_arr = np.array(batch_rewards) 
        dones_arr = np.array(batch_dones) 
        return obs_arr, rew_arr, dones_arr

    def compute_td_targets(self, rewards, dones, values, next_values):
        """
        Computes TD(λ) targets using bootstrapping.
        G_t = r_t + gamma * V(s_{t+1}) * (1 - done_t)
        """
        num_steps, num_envs = rewards.shape
        td_targets = np.zeros_like(rewards)

        next_value = next_values[-1]  # Final value to use for bootstrapping
        for t in reversed(range(num_steps)):
            mask = 1.0 - dones[t]
            td_targets[t] = rewards[t] + self.gamma * next_value * mask
            next_value = values[t]  # Move the next value pointer to the current state value

        return td_targets

    def compute_advantages(self, returns, predicted_values):
        """
        Computes the advantage by subtracting the predicted value from the returns.
        Then normalizes the advantages for stable training.
        """
        advantages = returns - predicted_values
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return advantages

    def train_step(self, num_steps=1024, batch_size=64, epochs=1):
        """
        Performs one full training cycle: Collect -> Calculate Returns -> Update Net
        """
        with torch.no_grad():
            obs, rewards, dones = self.collect_rollouts(num_steps)

        # Get predicted values from the value network
        values = self.value_net(torch.as_tensor(obs.reshape(-1, obs.shape[-1]), dtype=torch.float32).to(self.device))

        # Get the value estimates of next states (bootstrapping)
        next_obs = np.roll(obs, shift=-1, axis=0)  # Shift the observation for next state
        next_values = self.value_net(torch.as_tensor(next_obs.reshape(-1, obs.shape[-1]), dtype=torch.float32).to(self.device))

        # Compute TD(λ) targets using bootstrapping
        td_targets = self.compute_td_targets(rewards, dones, values.detach().cpu().numpy(), next_values.detach().cpu().numpy())
        # Normalize the advantages
        advantages = self.compute_advantages(td_targets.flatten(), values.detach().cpu().numpy())

        # Convert to torch tensors
        t_obs = torch.as_tensor(obs.reshape(-1, obs.shape[-1]), dtype=torch.float32).to(self.device)
        t_advantages = torch.as_tensor(advantages, dtype=torch.float32).to(self.device)
        t_td_targets = torch.as_tensor(td_targets.reshape(-1), dtype=torch.float32).to(self.device)

        dataset_size = len(t_obs)
        indices = np.arange(dataset_size)

        self.value_net.train()
        for _ in range(epochs):
            total_loss = 0
            np.random.shuffle(indices)
            for start in range(0, dataset_size, batch_size):
                end = start + batch_size
                idx = indices[start:end]
                batch_x = t_obs[idx]
                batch_adv = t_advantages[idx]
                batch_y = t_td_targets[idx]

                # Forward pass
                preds = self.value_net(batch_x)

                # Loss calculation (MSE loss between predicted values and TD targets)
                loss = self.loss_fn(preds, batch_y)

                # Backpropagation
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()

                total_loss += loss.item()

            if _ == 0:
                print(torch.mean(preds),torch.mean(batch_y))
        return total_loss / (epochs * (dataset_size // batch_size))

    def evaluate(self, num_episodes=20, device=None):
        """
        Evaluate the value function performance on multiple episodes.
        """
        if device is None:
            device = next(self.value_net.parameters()).device
        
        self.value_net.eval()
        predicted_values = []
        actual_returns = []

        with torch.no_grad():
            for ep in range(num_episodes):
                obs = self.env.reset()
                obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
                pred_value = self.value_net(obs_t)[0][0].item()
                predicted_values.append(pred_value)

                actual_return = 0
                discount = 1.0
                done = False

                while not done:
                    obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
                    action = self.policy.predict(obs)[0]
                    next_obs, reward, terminated, _ = self.env.step(action)
                    obs = next_obs
                    reward = reward[0]
                    done = terminated[0]

                    actual_return += discount * reward
                    discount *= self.gamma

                actual_returns.append(actual_return)

        predicted_values = np.array(predicted_values)
        actual_returns = np.array(actual_returns)

        mae = np.mean(np.abs(predicted_values - actual_returns))
        rmse = np.sqrt(np.mean((predicted_values - actual_returns) ** 2))
        correlation = np.corrcoef(predicted_values, actual_returns)[0, 1]

        print(f"Mean Absolute Error: {mae:.3f}")
        print(f"Root Mean Squared Error: {rmse:.3f}")
        print(f"Correlation: {correlation:.3f}")
        print(f"Predicted: {np.mean(predicted_values):.2f} ± {np.std(predicted_values):.2f}")
        print(f"Actual:    {np.mean(actual_returns):.2f} ± {np.std(actual_returns):.2f}")
        
        return mae, rmse, correlation


In [307]:
env = full_two_stage_env
policy = all_concept_model.policy
obs_dim = env.observation_space.shape[0]
my_value_net = ValueNet(obs_dim=env.observation_space.shape[0])
trainer = TDTrainer(my_value_net, env, policy, gamma=0.995)
for i in range(50):
    avg_loss = trainer.train_step()
    print(f"Iter {i}, Loss: {avg_loss:.4f}")
# trainer.evaluate()

tensor(0.0001, grad_fn=<MeanBackward0>) tensor(7.5466e-05)
Iter 0, Loss: 0.0008
tensor(9.3261e-06, grad_fn=<MeanBackward0>) tensor(2.2814e-05)
Iter 1, Loss: 0.0003
tensor(0.0014, grad_fn=<MeanBackward0>) tensor(9.8245e-05)
Iter 2, Loss: 0.0007
tensor(-0.0001, grad_fn=<MeanBackward0>) tensor(0.0006)
Iter 3, Loss: 0.0006
tensor(0.0019, grad_fn=<MeanBackward0>) tensor(0.0013)
Iter 4, Loss: 0.0009
tensor(0.0038, grad_fn=<MeanBackward0>) tensor(0.0021)
Iter 5, Loss: 0.0006
tensor(0.0047, grad_fn=<MeanBackward0>) tensor(0.0042)
Iter 6, Loss: 0.0004
tensor(0.0053, grad_fn=<MeanBackward0>) tensor(0.0048)
Iter 7, Loss: 0.0004
tensor(0.0074, grad_fn=<MeanBackward0>) tensor(0.0056)
Iter 8, Loss: 0.0006
tensor(0.0084, grad_fn=<MeanBackward0>) tensor(0.0109)
Iter 9, Loss: 0.0012
tensor(0.0091, grad_fn=<MeanBackward0>) tensor(0.0075)
Iter 10, Loss: 0.0006
tensor(0.0078, grad_fn=<MeanBackward0>) tensor(0.0079)
Iter 11, Loss: 0.0005
tensor(0.0082, grad_fn=<MeanBackward0>) tensor(0.0079)
Iter 12, Loss:

KeyboardInterrupt: 

Episode 1
Episode 2
Episode 3
Episode 4
Episode 5
Episode 6
Episode 7
Episode 8
Episode 9
Episode 10
Episode 11
Episode 12
Episode 13
Episode 14
Episode 15
Episode 16
Episode 17
Episode 18
Episode 19
Episode 20
Episode 21
Episode 22
Episode 23
Episode 24
Episode 25
Episode 26
Episode 27
Episode 28
Episode 29
Episode 30
Episode 31
Episode 32
Episode 33
Episode 34
Episode 35
Episode 36
Episode 37
Episode 38
Episode 39
Episode 40
Episode 41
Episode 42
Episode 43
Episode 44
Episode 45
Episode 46
Episode 47
Episode 48
Episode 49
Episode 50
Episode 1
Episode 2
Episode 3
Episode 4
Episode 5
Episode 6
Episode 7
Episode 8
Episode 9
Episode 10
Episode 11
Episode 12
Episode 13
Episode 14
Episode 15
Episode 16
Episode 17
Episode 18
Episode 19
Episode 20
0.10157694
Predicted vs Actual Correlation: 0.591


(array([0.92413503, 0.94531518, 0.94940591, ..., 0.9248063 , 0.9192608 ,
        0.922607  ]),
 array([0.95319998, 0.94843398, 0.94369181, ..., 0.91919109, 0.91459513,
        0.91002216]))

In [407]:
subset_concept, idx = policy_coverage_selection_lp_advantage(
    ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model,
    value_net,rollout_steps=250)

On rollout 0
On rollout 1
On rollout 2
On rollout 3
On rollout 4
On rollout 5
On rollout 6
On rollout 7
On rollout 8
On rollout 9
On rollout 10
On rollout 11
On rollout 12
On rollout 13
On rollout 14
On rollout 15
On rollout 16
On rollout 17
On rollout 18
On rollout 19
On rollout 20
On rollout 21
On rollout 22
On rollout 23
On rollout 24
On rollout 25
On rollout 26
On rollout 27
On rollout 28
On rollout 29
On rollout 30
On rollout 31
On rollout 32
On rollout 33
On rollout 34
On rollout 35
On rollout 36
On rollout 37
On rollout 38
On rollout 39
On rollout 40
On rollout 41
On rollout 42
On rollout 43
On rollout 44
On rollout 45
On rollout 46
On rollout 47
On rollout 48
On rollout 49
On rollout 50
On rollout 51
On rollout 52
On rollout 53
On rollout 54
On rollout 55
On rollout 56
On rollout 57
On rollout 58
On rollout 59
On rollout 60
On rollout 61
On rollout 62
On rollout 63
On rollout 64
On rollout 65
On rollout 66
On rollout 67
On rollout 68
On rollout 69
On rollout 70
On rollout 71
On

In [408]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,processed_concepts=processed_concepts,concept_idx=idx)
train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=150_000,custom_name="{}_perfect_greedy_{}".format(environment_string,seed)) 

approx_kl,▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▃▃▆▃▃▆▆██▆██▆▆▆▆█▅▅▅▅
clip_fraction,▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▁▁▁▁▄▄▃▃▃▂▂▂▂▂▆▆▆▆██▆▅▅▅▃
ema_norm_reward,▁▁▁▂▁▁▁▁▂▁▁▁▁▂▂▂▁▂▃▃▅▃▃▃▅▅▅▆▅▆▇▇▅▆▆▆▅█▆█
entropy_loss,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▃▄▄▅▅▅▆▆▆▆▆▆▆▆▆▆▇▇▇▇█
episode_length_mean,█████████████▇███▂██▁███▃▇▄█▄▃▇▃▅▇██▄▃▄▄
episode_reward_max,▁▁▁▁▁▁▁▁▁▁▁▁▁▆▁█▁▃▁▁▁▁▁█▃█▁▆▁▆▇▁▅▇▇▇█▅▃▇
episode_reward_mean,▁▁▁▁▁▁▁▁▄▁▆▁▁▁▃▂▁▁▁▁▅▁▁▂▁▃▅▁█▁▂▁▆▇█▆█▅▄▇
episode_reward_min,▁▁▁▁▁▃▁▁▁▁▁▁▁▇▁▁▁▆▇▄▁▁▂▄▁▁▄▇▁▆▄▆▂█▂▃▇▇▁▁
episodes_completed,▁▁▁▂▂▂▂▂▂▃▃▃▄▅▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇████
explained_variance,▁▁▁▁▁▅▅▅▅▅▅▅▅▆▆▇▇▆▇▇▇███▇▇▇███████████▇▇
+1,...


In [223]:
subset_concept, idx = policy_coverage_selection(
    ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model)

Coverage 0.99508507152533


In [224]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,processed_concepts=processed_concepts,concept_idx=idx)
all_concept_model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=150_000,custom_name="{}_perfect_greedy_{}".format(environment_string,seed)) 

approx_kl,▂▂▂▂▂▅▅▂▆▁▅▁▁▁▁▂▂▂▂▂▃▃▃▅▅████▁▁▁▁▁▁▄▅▅▅█
clip_fraction,▁▁▂▂▂▄▄▄▄▃▃▁▁▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▁██▁▁▁▁▃███▇
ema_norm_reward,▁▁▂▁▁▂▁▁▃▂▄▅▂▂▁▂▃▅▄▃▄▃▂▂▂▅▅▃▆▄▂▄▆▇▇▅▆▆▆█
entropy_loss,▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▅▅▅▅▆▇▇▇▇▇▇▇▇▇▇▇▇██
episode_length_mean,███████████▅██▆█▆██▇▅▄████▅█▁██▅██████▆▅
episode_reward_max,▁▁▁▁▁▁▁▁▁▁▁▁▆▂▁▄▃▁▁▁▁▁▁█▁▅▃▄▆▅▁▇▁▁▅▁▆▅▁▄
episode_reward_mean,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄▁▁▆▁▁▃▁▁▁▁▁▇▁▆▆▁▆▁▁█▁▂
episode_reward_min,▁▁▁▁▁▁▁▁▁▄▁▁▁▁▁▁▁▁▃▁▃▁▁▁▃▁▄█▆▁▁▁▅▆▁▆▁▁▄▁
episodes_completed,▁▁▁▂▂▃▃▃▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇██
explained_variance,▁▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█████████▇▇███▇█████████
+1,...


In [225]:
subset_concept, idx = policy_coverage_selection_lp(
    ground_truth_gym_env,
    concept_list,
    num_concepts_selected,
    groundtruth_model)

Coverage 0.9956


In [226]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,processed_concepts=processed_concepts,concept_idx=idx)
all_concept_model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=150_000,custom_name="{}_perfect_greedy_{}".format(environment_string,seed)) 

approx_kl,▁▁▁▁▂▄▄▃▃▃██▅▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▄▇▇▇▄▄▄▄▄▆▃
clip_fraction,▁▁▁▁▁▁▁▁▁▁▂▂▂▁▁█████▂▂▂▁▁▂▂▄▄▄▂▂▂▂▂▅▄▄▄▄
ema_norm_reward,▂▂▁▁▂▂▃▁▁▁▃▃▅▃▂▄▃▄▄▃▄▅▅▃▄▅▅▅▅▆▅▃▅▅▅▆▇▇▆█
entropy_loss,▁▁▁▁▁▁▁▁▂▂▂▂▃▃▃▃▄▄▅▅▆▇▇▇███▇▇▇▇▆▆▆▆▆▆▆▇▇
episode_length_mean,███▁████████▄▇▆▆▇▅████▅█████▄▄██████▄███
episode_reward_max,▁▁▁▁▁▂▁▁▂▁▁▁▃▁▁▃▁▅▄▁▂▁▁▁▃▄▁▆▁▂▃▆▅▁██▆▇▃▅
episode_reward_mean,▄▁▁▁▁▅▃▂▂▇▁▁▂▇▁▃▂▃▄▁▁▃▄▅▁▂▄▃▁▃▃▄▁▃█▂▁█▁▁
episode_reward_min,▁▆▁▁▄▁▁▁█▄▆▃▁▁▁▅▁▁▅▃▆▁▁▄▁▁▁▁▁▁▁▃▅█▅▁▇▁▁▄
episodes_completed,▁▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇█
explained_variance,▁▆▆▆▆▆▆▆▆▆▆█▇▇▇▇▇████████▇▇▇▇▇▇▇▇▇▇▇▇▇▇█
+1,...


In [227]:
subset_concept, idx = policy_coverage_selection_exp_lp(
    ground_truth_gym_env,
    concept_list,
    [1 for i in range(len(concept_list))],
    num_concepts_selected,
    groundtruth_model)

In [228]:
two_stage_env, two_stage_gym_env = get_environment(environment_string,concept_list,seed,processed_concepts=processed_concepts,concept_idx=idx)
all_concept_model = train_ppo_model(two_stage_env,environment_string,policy="MlpPolicy",total_timesteps=150_000,custom_name="{}_perfect_greedy_{}".format(environment_string,seed)) 

approx_kl,▁▁▁▃▃▃▃▃▂▃▃▃▃▃▃▃▃▆▆▆███▃▃▆▆▆▆▄▄▆▆▆▆▃▃▇▇▂
clip_fraction,▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▁▇▇▇▇▇▁▁▅▅▆▆▂▂▂▄▁▁▁▁▂██▁▁
ema_norm_reward,▃▁▁▁▁▂▂▃▃▃▃▃▄▃▃▄▅▆▇▅▃▆▅▅▆▆▄▅▅▅▇▆▅▆▆▇███▆
entropy_loss,▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▃▃▄▄▄▄▅▅▅▆▆▆▆▅▅▆▆▆▇▇▇███
episode_length_mean,█████▁███▇████▂▂█▃▅███▇▆█▃█▁▄█▂████▁▂▁█▂
episode_reward_max,▁▁▁▁▁▆▁▆▄▁▁▁▁▄▁▁▄▆█▆▆██▇▆▂▄▁▆▃▁▇▃▆█▆▄▁▄▃
episode_reward_mean,▁▂▁▁▁▁▁▁▇▁▁▃▁▁▁▆▂▁▆▁▁▄▃▅▁▅▁▃▆▁▁▁▇▅▆█▁▆▇▆
episode_reward_min,▁▁▁▁▂▃▁▁▁▁▁▁▁▁▁▁▃█▁▇▅▆▃▁▅▁▅▄▁▃▁▇▁▇▁▅▇▇▃▇
episodes_completed,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▃▃▄▄▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇█████
explained_variance,▁▁▁▅▅▃▄▄▄▄▆▆▆▇▇▇████████▇▇█▇▇▇██▇▇▇▇▇▇▇▇
+1,...
